# Figure S2 — Integration of the four technologies enables cell-type composition comparison and throughput selection

Every panel here is a *measurement on a file that exists before the figure is
drawn*. This notebook shows, for each panel, which file goes in and the shortest
complete code that produces the numbers and the plot. Journal layout (panel
letters, the 18.0 × 17.6 cm page, shared colour key, compositing) is done
separately and is not reproduced here.

| Panel | What it shows | Input | Code |
|---|---|---|---|
| A | WNN UMAP of 70,643 nuclei in four facets, points coloured by library over a grey shadow of all cells | `3k_dsci_ht_10x_gs_subset.RDS`, reduction `wnn.umap` + `orig.ident` | R/Seurat (§2) |
| B | stacked bars: 33-type composition of each technology (10x / dsciCAT-seq / HT-scCAT-seq-HT) | the same object, `@meta.data` only | R/ggplot2 (§3) |
| C | NMI and ARI against down-sampled throughput, 8 levels × 20 seeds | `3k_dsci_full_annotCOP.RDS` → §4 | R/Seurat + harmony (§4–§5) |
| D | heatmap of per-type recall, 37 types × 8 levels, types that improve with depth marked | the same §4 output | R (§6) |

```
A-B: integrated three-technology atlas — taken as given, this notebook starts here
3k_dsci_ht_10x_gs_subset.RDS  (70,643 nuclei; RNA+ATAC, harmony WNN; built by
                                 Figure 2's notebook from the per-technology matrices)
   reductions umap / umap.atac / wnn.umap ; orig.ident = 4 libraries ; cell_type (32)
        │
        ├─ §1b  derive cell_type_updated (RNA_snn_res.1 == 34 -> COP/NFOL, 280 cells,
        │        33 types) + technique (50k+100k -> dsciCAT-seq); the fixed type order
        │        dot_order, its numbering num_map and the shared palette cols
        ├─ §2   panel A: facet UMAP on a grey shadow of all 70,643 cells
        └─ §3   panel B: stacked composition per technique, 33 types

C-D: the throughput experiment, run on the dsciCAT-seq libraries alone
3k_dsci_full.RDS  (59,962 cells = FY13 21,038 + FY14 38,924; 36 types, COP/NFOL still
                   inside Oligo)  ->  the 37-type truth object is derived in §1a
        │
        ├─ §1a  cell-level split  old Oligo & (Gpr17 > 0 | Bmp4 > 0) -> COP/NFOL
        │        = 269 cells (0.45%), 37 types   ->  3k_dsci_full_annotCOP.RDS
        ├─ §4   8 levels x 20 seeds, each seed independent:
        │        subset -> NormalizeData -> 3000 HVG -> ScaleData -> PCA(50)
        │             -> RunHarmony(orig.ident) -> FindNeighbors(dims 1:50)
        │             -> FindClusters(Leiden, resolution 1)
        │        -> {level}_ARI_NMI.csv          (majority-vote labels vs truth)
        ->      {level}_per_seed_per_type.csv    (cluster x truth contingency)
        ├─ §5   panel C: mean ± SD over the 20 seeds, x = cells analysed
        └─ §6   panel D: mean recall per type per level, x ordered by full-data
                 abundance (rare first); per-seed paired Wilcoxon 40k vs 3k,
                 BH-FDR < 0.05 and Δrecall >= 0.5 -> marked
```

§4 is the only part that is expensive, and it is the part the figure's claim
rests on: *the truth label of every down-sampled cell is the label it already
had in the full atlas*, so 160 clustering runs need no manual annotation and the
per-type numbers cannot be fudged by a majority-vote relabelling.

---

## 0. Environment, inputs, outputs

| Variable | Default | Contents |
|---|---|---|
| `S2_DATA` | `<work>/FY` | the two Seurat objects + `3k_dsci/` (truth CSVs, §4 cache); layout in §0.1 |
| `S2_OUT` | `figS2_out` | created here; every panel PDF / CSV / cache lands here |
| `S2_LEVELS` | all 8 | comma list to restrict §4 (`3k,5k`) — for a smoke run |
| `S2_SEEDS` | `seq(1,200,by=10)` | how many repeats; shortening it changes §5/§6 error bars |
| `S2_NCORES` | per level (20 → 4) | §4 parallelism, capped by RAM not by cores |
| `S2_FORCE` | unset | `1` re-runs §4 even when the cache exists |
| `S2_BUILD_TRUTH` | unset | `1` re-derives `3k_dsci_full_annotCOP.RDS` in §1a (needs ~30 GB RAM) |

R ≥ 4.1 with `Seurat`, `harmony`, `ggplot2`, `dplyr`, `tidyr`, `stringr`, `pals`,
`parallel`; `aricode` for NMI/ARI (§4 falls back to an embedded implementation
verified against it) and `ggtext` for the coloured axis labels of panel D.

> ⚠ **This notebook has not been executed in the environment it was written
> in** — `aricode` and `ggtext` are absent there and there is no network to
> install them, and a full §4 re-run is ~3 h. Every number quoted below was
> read or recomputed from the CSVs the delivered figure was drawn from (the
> table is in `README.md`). Run it once where those two packages exist before
> treating the notebook as executed.

In [1]:
import matplotlib
matplotlib.use('Agg')
import os, subprocess, time, json, glob
import numpy as np

DATA = os.environ.get('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  = os.environ.get('S2_OUT', 'figS2_out')
os.makedirs(OUT, exist_ok=True)
print('DATA =', DATA, '| exists:', os.path.isdir(DATA), '| OUT =', OUT)

def show(*paths):
    # report the saved file only -- nothing is displayed inline (section 0.2)
    for q in paths:
        print('%s  (%.0f KB)' % (q, os.path.getsize(q) / 1024.0))


def run_r(rscript, env_extra=None, echo=True):
    '''run one R script file; return (returncode, text)'''
    env = dict(os.environ)
    if env_extra:
        env.update(env_extra)
    t0 = time.time()
    r = subprocess.run(['Rscript', rscript], env=env, capture_output=True, text=True)
    dt = (time.time() - t0) / 60
    if echo:
        print(r.stdout[-4000:])
        if r.stderr.strip():
            print('--- stderr (tail) ---')
            print(r.stderr[-1500:])
    print('Rscript exit code %d - %.1f min' % (r.returncode, dt))
    return r.returncode

DATA = <work>/FY | exists: True | OUT = <repo>/figS2/figS2_out


### 0.1 Input files

| Used by | Path (relative to `S2_DATA`) |
|---|---|
| §1b, §2, §3 (panels A–B) | `3k_dsci_ht_10x_gs_subset.RDS` — WNN-integrated Seurat object, 70,643 nuclei, mouse brain (13.2 GB) |
| §1a (truth of the throughput experiment) | `3k_dsci/3k_dsci_full.RDS` — dsciCAT-seq only, 59,962 cells, 36 types with COP/NFOL inside Oligo (5.4 GB) |
| §1a verification without loading 5.4 GB | `3k_dsci/annotCOP_marker_df.rds` — one row per cell: `cell_type`, `cell_type_wnn`, `rna_cluster`, `wnn_cluster`, `Gpr17`, `Bmp4`, `Plp1` |
| §4 (cache / skip) | `3k_dsci/3k_dsci_down{level}_ARI_NMI.csv` + `3k_dsci/{level}_per_seed_per_type.csv` |
| §1a, §6 (x-axis order of panel D) | `3k_dsci/3k_dsci_full_annotCOP_meancelltype.csv` — `type,Freq` of the 37 types |

`orig.ident` of the integrated object carries four libraries
(`10x`, `HT-scCAT-seq-HT`, `dsciCAT-seq-50k`, `dsciCAT-seq-100k`);
`orig.ident` of the dsciCAT-only object carries two (`FY13`, `FY14`), which is
what §4's harmony step regresses out.

### 0.2 Provenance — read this before comparing panels

* **Panels A–B and panels C–D are computed on different cell sets, and the
  figure says so in different words.** A–B use the integrated three-technology
  atlas (70,643 nuclei, 33 types, COP/NFOL = 280 cells = 0.4 %). C–D use the
  dsciCAT-seq libraries alone (59,962 cells, 37 types, COP/NFOL = 269 = 0.45 %):
  the question panel C–D answer is "how many cells of *our* library do I need to
  pay for", which is only meaningful on the deep single-technology data. The two
  COP/NFOL counts differ because the two objects differ, not because one of them
  is stale.
* **Both objects start from an annotation made upstream.** The RNA+ATAC harmony
  WNN integration, the clustering and the 32/36-type annotation are production
  steps and are not recomputed here. What *is* recomputed is the one relabelling
  the figure depends on: COP/NFOL. In the integrated object it is a
  cluster-level statement (`RNA_snn_res.1 == 34` → COP/NFOL, §1b); in the
  dsciCAT object it is a cell-level marker rule (`old Oligo & (Gpr17 > 0 |
  Bmp4 > 0)`, §1a) so that it survives every down-sample. Both give ~0.4 % and
  both are applied here from the stored data, not from a hand-edited table.
* **NMI depends on the variant — this changed panel C's numbers.** `aricode::NMI`
  defaults to `variant = "max"` (MI / max(H(x), H(y))), not the more familiar
  MI / √(H(x)H(y)). The 3k value is **0.726 with "max"** and 0.806 with "sqrt"
  (ARI is unaffected: 0.652 either way). The delivered panel C is the "max"
  series. §4 therefore uses `aricode` when installed and otherwise the embedded
  implementation, which was checked against the package itself on simulated
  partitions: **ARI identical (diff = 0), NMI max-variant diff ≈ 1e-16**.
* **20 seeds are not a formality, they are the point.** A rare type's
  "detected" flag at a single seed is a coin flip — measured in the first
  single-seed pass: COP/NFOL was recovered at 5k (15 of 22 cells in a 21-cell
  cluster, plurality COP/NFOL) and *not* recovered at 10k (all 21 cells absorbed
  into a 1,008-cell Oligo cluster). Both runs are legitimate; the type sits at
  0.45 %, exactly on the resolution boundary. Averaging 20 seeds turns that
  jitter into the monotone curves of panels C and D. The same seed list
  (`seq(1, 200, by = 10)`) is used at every level, which is what makes the
  per-seed paired test of §6 valid.
* **"Recall" here is computed from the contingency table, not from a relabelling.**
  For each type: recall = (cells of that type whose cluster has that type as
  plurality) / (cells of that type sampled). A cluster is allowed to be impure —
  majority-vote relabelling of the clusters is used *only* for the global
  NMI/ARI of panel C, exactly as in the original analysis, so the two panels
  answer two different questions and are not interchangeable.
* **A type that is never drawn in a seed produces no row at all** (recall is
  undefined, not zero). Measured: the 3k CSV has 739 rows, not 740 — `Ex_L4`
  (134 cells in full, expected 6.7 at 3k) was missed entirely in one of the 20
  seeds. Panel D shows those cells in `grey92`.
* **Cost.** §4 as delivered took ≈ 3 h on a 96-core host (≈ 12 min for 3k,
  ≈ 20 min per level, 40k the slowest). Parallelism is capped by memory, not by
  cores: 20 workers at 3k down to 4 at 40k, with the RNA-only assays kept and
  the rest dropped after loading. `S2_LEVELS`/`S2_SEEDS` give a cheap smoke run.
* **Production touches not reproduced here.** The delivered page sets Arial
  (fontconfig alias → Liberation Sans, embedded through `cairo_pdf`), 6–8 pt
  text, the white-padded panel letters A–D, the exact 18.0 × 17.6 cm geometry,
  and the red colouring of the 16 significant type labels via `ggtext`. Panel
  sizes below are the delivered slot sizes so the output can be dropped into
  that layout; the red labels degrade to a `+` suffix when `ggtext` is absent.

### 0.3 Input check

In [2]:
R0 = r'''
suppressPackageStartupMessages({library(Seurat); library(ggplot2); library(dplyr)})
DATA <- Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)
need <- c('3k_dsci_ht_10x_gs_subset.RDS', '3k_dsci/3k_dsci_full.RDS',
          '3k_dsci/annotCOP_marker_df.rds', '3k_dsci/3k_dsci_full_annotCOP.RDS',
          '3k_dsci/3k_dsci_full_annotCOP_meancelltype.csv')
for (f in need) cat(sprintf('%-52s %s  %6.2f GB\n', f,
    file.exists(file.path(DATA, f)), ifelse(file.exists(file.path(DATA, f)),
    file.info(file.path(DATA, f))$size / 1e9, NA)))
lv <- c('3k','5k','10k','15k','20k','25k','30k','40k')
cat('\n§4 cache present (skipped when both CSVs of a level exist):\n')
print(data.frame(level = lv,
                 ari_nmi = file.exists(file.path(DATA, '3k_dsci',
                                                 paste0('3k_dsci_down', lv, '_ARI_NMI.csv'))),
                 per_type = file.exists(file.path(DATA, '3k_dsci',
                                                  paste0(lv, '_per_seed_per_type.csv')))))
cat('\noptional packages:',
    paste(sapply(c('aricode', 'ggtext'), function(p)
        sprintf('%s=%s', p, requireNamespace(p, quietly = TRUE))), collapse = '  '), '\n')
'''
r0 = os.path.join(OUT, 'figS2_check.R')
with open(r0, 'w') as fh:
    fh.write(R0)
run_r(r0)

3k_dsci_ht_10x_gs_subset.RDS                         TRUE   13.19 GB
3k_dsci/3k_dsci_full.RDS                             TRUE    5.41 GB
3k_dsci/annotCOP_marker_df.rds                       TRUE    0.00 GB
3k_dsci/3k_dsci_full_annotCOP.RDS                    TRUE    5.41 GB
3k_dsci/3k_dsci_full_annotCOP_meancelltype.csv       TRUE    0.00 GB

§4 cache present (skipped when both CSVs of a level exist):
  level ari_nmi per_type
1    3k    TRUE     TRUE
2    5k    TRUE     TRUE
3   10k    TRUE     TRUE
4   15k    TRUE     TRUE
5   20k    TRUE     TRUE
6   25k    TRUE     TRUE
7   30k    TRUE     TRUE
8   40k    TRUE     TRUE

optional packages: aricode=TRUE  ggtext=TRUE 

Rscript exit code 0 - 0.4 min


0

**Expected** — all five inputs present; the §4 cache present or absent (absent
just means §4 will actually run); `aricode=TRUE` strongly preferred for §4 (see
§0.2 for why the variant matters), `ggtext=TRUE` only affects how panel D marks
its significant labels.

### 0.4 Software

Printed here rather than at the end so a run can be pinned to one environment.

In [3]:
R4 = r'''
cat('R', R.version.string, '\n')
pkgs <- c('Seurat','SeuratObject','harmony','ggplot2','dplyr','tidyr','stringr',
          'pals','parallel','aricode','ggtext','Matrix','Rcpp')
for (p in pkgs) cat(sprintf('%-14s %s\n', p,
    tryCatch(as.character(packageVersion(p)), error = function(e) 'NOT INSTALLED')))
'''
r4 = os.path.join(OUT, 'figS2_versions.R')
with open(r4, 'w') as fh:
    fh.write(R4)
run_r(r4)
import sys, platform
print('python', sys.version.split()[0], '|', platform.platform())
for m in ['numpy', 'pandas', 'matplotlib']:
    try:
        mod = __import__(m); print('%-12s %s' % (m, mod.__version__))
    except ImportError:
        print('%-12s not installed' % m)

R R version 4.2.2 (2022-10-31) 
Seurat         4.3.0
SeuratObject   4.1.3
harmony        1.2.3
ggplot2        3.5.2
dplyr          1.2.1
tidyr          1.3.2
stringr        1.5.1
pals           1.8
parallel       4.2.2
aricode        1.0.1
ggtext         0.2.0
Matrix         1.5.3
Rcpp           1.0.14

Rscript exit code 0 - 0.0 min
python 3.8.14 | Linux-5.15.0-130-generic-x86_64-with-glibc2.10
numpy        1.22.4
pandas       1.5.1
matplotlib   3.6.2


---

## 1. The truth annotations — two objects, one label each

### 1a. The dsciCAT-only object: split COP/NFOL out of Oligo at cell level

**Input**: `3k_dsci/3k_dsci_full.RDS` (59,962 cells, 36 types) or, to check the
rule without loading 5.4 GB, `3k_dsci/annotCOP_marker_df.rds`.
**Output**: `S2_OUT/truth_annotCOP.csv` (`cell, cell_type, rna_cluster`) — the
label vector §4 draws its down-samples against — and, with `S2_BUILD_TRUTH=1`,
`3k_dsci_full_annotCOP.RDS` itself.

The rule is deliberately cell-level and marker-based, because it has to survive
160 independent down-samples: a cell is COP/NFOL if it was annotated Oligo and
expresses Gpr17 or Bmp4. In the full object this separates 269 cells (0.45 %)
into a 37th type; the mature-Oligo clusters (Gpr17 < 2 %) are untouched.

In [4]:
R1A = r'''
suppressPackageStartupMessages({library(Seurat)})
DATA <- Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
FY   <- file.path(DATA, '3k_dsci')
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

suppressPackageStartupMessages({library(Seurat)})
DATA <- Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
FY   <- file.path(DATA, '3k_dsci')
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

# ---- the rule, derived from the ORIGINAL object (cell_type has 36 types,     --------
#      no COP/NFOL yet), with the compact marker table as a cross-check only.   --------
#      annotCOP_marker_df.rds is a POST-split export (its cell_type already     --------
#      contains the 269 COP/NFOL) -- using it as the derivation input silently  --------
#      yields 0 splits and "passes" because the composition is unchanged.       --------
seu <- readRDS(file.path(FY, '3k_dsci_full.RDS'))
mdf <- readRDS(file.path(FY, 'annotCOP_marker_df.rds'))
stopifnot(ncol(seu) == nrow(mdf))                       # same cell set
rna <- GetAssayData(seu, assay = 'RNA', slot = 'data')
g <- as.numeric(rna['Gpr17', ]); b <- as.numeric(rna['Bmp4', ])
old_oligo <- as.character(seu$cell_type) == 'Oligo'
cop <- old_oligo & (g > 0 | b > 0)
new <- as.character(seu$cell_type); new[cop] <- 'COP/NFOL'
cat('cells:', ncol(seu), '| old Oligo:', sum(old_oligo),
    '| split to COP/NFOL:', sum(cop),
    sprintf('(%.2f%%)', 100 * mean(cop)), '
')
cat('types after split:', length(unique(new)), '
')
cat('COP/NFOL by RNA cluster:',
    paste(names(table(seu$RNA_snn_res.1[cop])), table(seu$RNA_snn_res.1[cop]),
          sep = '=', collapse = ', '), '
')
cat(sprintf('within old Oligo: Gpr17>0 %.1f%% | Bmp4>0 %.1f%% | both %.1f%%
',
            100 * mean(g[old_oligo] > 0), 100 * mean(b[old_oligo] > 0),
            100 * mean(g[old_oligo] > 0 & b[old_oligo] > 0)))
# cross-check against the post-split marker export: same cells, same labels
mk <- match(mdf$cell, colnames(seu))
stopifnot(identical(as.character(mdf$cell_type[mk]), new))
cat('marker-df cross-check: identical labels for all', nrow(mdf), 'cells
')
write.csv(data.frame(cell = colnames(seu), cell_type = new,
                     rna_cluster = as.character(seu$RNA_snn_res.1)),
          file.path(OUT, 'truth_annotCOP.csv'), row.names = FALSE)

# ---- composition of the truth object (panel D's x order comes from this) ---
comp <- sort(table(new), decreasing = TRUE)
write.csv(data.frame(type = names(comp), Freq = as.integer(comp)),
          file.path(OUT, 'truth_composition.csv'), row.names = FALSE)
fdel <- file.path(FY, '3k_dsci_full_annotCOP_meancelltype.csv')
if (!file.exists(fdel)) stop('delivered composition table missing: ', fdel)
deliv <- read.csv(fdel)
stopifnot(nrow(deliv) == length(comp), sum(deliv$Freq) == sum(comp),
          identical(sort(deliv$type), sort(names(comp))))
cat('composition matches the delivered table (37 types,', sum(comp), 'cells)
')

# ---- optionally rebuild the RDS the experiment runs on --------------------
if (Sys.getenv('S2_BUILD_TRUTH') == '1') {
  idx <- which(cop)                       # derived above, no second 5.4 GB read
  ct <- as.character(seu$cell_type); ct[idx] <- 'COP/NFOL'
  seu$cell_type <- factor(ct)
  if (!is.null(seu$cell_type_wnn)) {
    cw <- as.character(seu$cell_type_wnn)
    ow <- cw == 'Oligo'
    cw[ow & (g > 0 | b > 0)] <- 'COP/NFOL'
    seu$cell_type_wnn <- factor(cw)
  }
  saveRDS(seu, file.path(OUT, '3k_dsci_full_annotCOP.RDS'))
  cat('wrote', file.path(OUT, '3k_dsci_full_annotCOP.RDS'),
      '| COP/NFOL:', sum(ct == 'COP/NFOL'), '
')
}

# ---- composition of the truth object (panel D's x order comes from this) ---
comp <- sort(table(new), decreasing = TRUE)
write.csv(data.frame(type = names(comp), Freq = as.integer(comp)),
          file.path(OUT, 'truth_composition.csv'), row.names = FALSE)
fdel <- file.path(FY, '3k_dsci_full_annotCOP_meancelltype.csv')
if (!file.exists(fdel)) stop('delivered composition table missing: ', fdel)
deliv <- read.csv(fdel)
stopifnot(nrow(deliv) == length(comp), sum(deliv$Freq) == sum(comp),
          identical(sort(deliv$type), sort(names(comp))))
cat('composition matches the delivered table (37 types,', sum(comp), 'cells)\n')

# ---- optionally rebuild the RDS the experiment runs on --------------------
if (Sys.getenv('S2_BUILD_TRUTH') == '1') {
  seu <- readRDS(file.path(FY, '3k_dsci_full.RDS'))
  stopifnot(ncol(seu) == nrow(mdf))
  rna <- GetAssayData(seu, assay = 'RNA', slot = 'data')
  g <- as.numeric(rna['Gpr17', ]); b <- as.numeric(rna['Bmp4', ])
  idx <- which(as.character(seu$cell_type) == 'Oligo' & (g > 0 | b > 0))
  ct <- as.character(seu$cell_type); ct[idx] <- 'COP/NFOL'
  seu$cell_type <- factor(ct)
  if (!is.null(seu$cell_type_wnn)) {
    cw <- as.character(seu$cell_type_wnn)
    ow <- cw == 'Oligo'
    cw[ow & (g > 0 | b > 0)] <- 'COP/NFOL'
    seu$cell_type_wnn <- factor(cw)
  }
  saveRDS(seu, file.path(OUT, '3k_dsci_full_annotCOP.RDS'))
  cat('wrote', file.path(OUT, '3k_dsci_full_annotCOP.RDS'),
      '| COP/NFOL:', sum(ct == 'COP/NFOL'), '\n')
}
'''
r1a = os.path.join(OUT, 'figS2_s1a_truth.R')
with open(r1a, 'w') as fh:
    fh.write(R1A)
run_r(r1a)

cells: 59962 | old Oligo: 6338 | split to COP/NFOL: 269 (0.45%) 
types after split: 37 
COP/NFOL by RNA cluster: 0=0, 1=133, 10=0, 11=0, 12=0, 13=0, 14=0, 15=0, 16=0, 17=0, 18=0, 19=0, 2=0, 20=0, 21=0, 22=0, 23=0, 24=0, 25=0, 26=0, 27=0, 28=0, 29=0, 3=0, 30=0, 31=0, 32=0, 33=135, 34=0, 35=0, 36=0, 37=0, 38=0, 39=1, 4=0, 40=0, 41=0, 42=0, 5=0, 6=0, 7=0, 8=0, 9=0 
within old Oligo: Gpr17>0 3.7% | Bmp4>0 1.3% | both 0.7%
marker-df cross-check: identical labels for all 59962 cells
composition matches the delivered table (37 types, 59962 cells)
composition matches the delivered table (37 types, 59962 cells)

--- stderr (tail) ---
Loading required package: Signac

Rscript exit code 0 - 1.9 min


0

**Expected** — 59,962 cells; 269 split to COP/NFOL (0.45 %); 37 types; the
composition table matches the delivered `3k_dsci_full_annotCOP_meancelltype.csv`
exactly (checked with `stopifnot` above). With `S2_BUILD_TRUTH=1` the rebuilt RDS
is written to `S2_OUT` rather than over the analysis tree.

### 1b. The integrated object: the 33-type annotation panels A–B use

**Input**: `3k_dsci_ht_10x_gs_subset.RDS` (13.2 GB).
**Output**: `S2_OUT/s2_light.RDS` — the small derived cache panels A–B consume
(`wnn.umap` embeddings + `orig.ident`/`technique`/`cell_type_updated` + the shared
order, numbering and palette) — and every counted number, printed.

How the per-technology matrices become this object is Figure 2's notebook; this
one starts from it and only re-derives the annotation labels A/B plot — the same
derivation Figure 2 makes, repeated here so Figure S2 is self-contained. The read
is the only expensive step of §1b; everything after it uses `s2_light.RDS`.

In [5]:
R1B = r'''
suppressPackageStartupMessages({library(Seurat); library(pals)})
DATA <- Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)
t0 <- proc.time()[3]
obj <- readRDS(file.path(DATA, '3k_dsci_ht_10x_gs_subset.RDS'))
cat('loaded in', round((proc.time()[3] - t0) / 60, 1), 'min |', ncol(obj), 'cells\n')

md <- obj@meta.data
md$cell_type_updated <- as.character(md$cell_type)
md$cell_type_updated[md$RNA_snn_res.1 == 34] <- 'COP/NFOL'      # 280 cells
md$technique <- ifelse(md$orig.ident %in% c('dsciCAT-seq-50k', 'dsciCAT-seq-100k'),
                       'dsciCAT-seq', as.character(md$orig.ident))
cat('libraries:\n'); print(table(md$orig.ident))
cat('techniques:\n'); print(table(md$technique))
cat('types:', length(unique(md$cell_type_updated)),
    '| COP/NFOL:', sum(md$cell_type_updated == 'COP/NFOL'),
    sprintf('(%.2f%%)', 100 * mean(md$cell_type_updated == 'COP/NFOL')), '\n')
stopifnot(ncol(obj) == 70643, length(unique(md$cell_type_updated)) == 33)

# fixed 33-type order + numbering + shared palette (= Fig 2's, = track/dot plot)
dot_order <- c('AST','Oligo','COP/NFOL','OPC','MG','EC','VLMC','Ependyma',
               'In_D1','In_D2','In_HAP1','In_LAMP5','In_LHX6','In_SST','In_VIP',
               'In_PVALB','In_TH','Neuron_MB','Ex_LHA','Ex_HY-MM','Ex_DG','Ex_TH',
               'Ex_MH-LH','Ex_SUB_Pros','Ex_CA1_Pros','Ex_CAR3','Ex_L2/3_IT',
               'Ex_L2/3_IT_PIR','Ex_L4/5_IT','Ex_L5/6_NP','Ex_L6_IT','Ex_L6_CT','Ex_L6b')
num_map <- setNames(seq_along(dot_order), dot_order)
all_types_full <- sort(unique(c(
  'AST','COP/NFOL','EC','Ependyma','Ex_CA1_Pros','Ex_CA3','Ex_CAR3','Ex_CR',
  'Ex_DG','Ex_HY-MM','Ex_IC','Ex_L2/3_IT','Ex_L2/3_IT_PIR','Ex_L4','Ex_L4/5_IT',
  'Ex_L5_ET','Ex_L5_IT','Ex_L5_NP','Ex_L5/6_NP','Ex_L6_CT','Ex_L6_IT','Ex_L6b',
  'Ex_LHA','Ex_MH-LH','Ex_SUB_Pros','Ex_TH','In_D1','In_D2','In_D1-D2','In_HAP1',
  'In_LAMP5','In_LHX6','In_MEIS2','In_PVALB','In_SST','In_TH','In_VIP','MG',
  'Neuron_MB','Oligo','OPC','VLMC')))
global_colors_full <- unique(c(as.vector(pals::glasbey(32)),
                               as.vector(pals::alphabet(26))))[1:length(all_types_full)]
names(global_colors_full) <- all_types_full
stopifnot(all(dot_order %in% unique(md$cell_type_updated)))

saveRDS(list(wnn = SeuratObject::Embeddings(obj, 'wnn.umap'), meta = md,
             dot_order = dot_order, num_map = num_map,
             cols = global_colors_full[dot_order]),
        file.path(OUT, 's2_light.RDS'))
cat('wrote s2_light.RDS\n')
'''
r1b = os.path.join(OUT, 'figS2_s1b_object.R')
with open(r1b, 'w') as fh:
    fh.write(R1B)
run_r(r1b)

loaded in 3.9 min | 70643 cells
libraries:

             10x dsciCAT-seq-100k  dsciCAT-seq-50k  HT-scCAT-seq-HT 
            8438            30350            17093            14762 
techniques:

            10x     dsciCAT-seq HT-scCAT-seq-HT 
           8438           47443           14762 
types: 33 | COP/NFOL: 280 (0.40%) 
wrote s2_light.RDS

--- stderr (tail) ---
Loading required package: Signac

Rscript exit code 0 - 4.6 min


0

**Expected** — 70,643 cells; `10x` 8,438 · `HT-scCAT-seq-HT` 14,762 ·
`dsciCAT-seq-50k` 17,093 · `dsciCAT-seq-100k` 30,350, i.e. techniques
dsciCAT-seq 47,443 / HT-scCAT-seq-HT 14,762 / 10x 8,438; 33 types; COP/NFOL 280
(0.4 %) — the legend numbers of panels A and B. The load is 4.5 min, the whole
step 5 min.

---

## 2. Panel A — the four libraries on one shared WNN embedding

**Input**: `wnn` + `orig.ident` from the §1b cache.
**Output**: `S2_OUT/S2a_batch_shadow_facets.pdf` (+ `.png`).

Four facets, one per library. Each facet draws *all* 70,643 nuclei in grey
(the shadow) and its own library's cells in colour on top, so a library that
occupies a region of the atlas is visibly interleaved with the others rather
than sitting in its own corner. Point radius is small but printable
(`size = 0.15`, ≈ 0.4 pt) — the original analysis figure used 0.02/0.03, whose
points are ≈ 0.05 pt and vanish in print; that is why this panel was redrawn.

In [6]:
R2 = r'''
suppressPackageStartupMessages({library(ggplot2)})
OUT <- Sys.getenv('S2_OUT', 'figS2_out')
S   <- readRDS(file.path(OUT, 's2_light.RDS'))
co  <- as.data.frame(S$wnn); colnames(co) <- c('UMAP_1', 'UMAP_2')
batch_order  <- c('10x', 'HT-scCAT-seq-HT', 'dsciCAT-seq-50k', 'dsciCAT-seq-100k')
batch_colors <- c('10x' = '#E64B35FF', 'HT-scCAT-seq-HT' = '#3C5488FF',
                  'dsciCAT-seq-50k' = '#00A087FF', 'dsciCAT-seq-100k' = '#4DBBD5FF')
co$Batch <- factor(as.character(S$meta$orig.ident), levels = batch_order)
stopifnot(!any(is.na(co$Batch)), nrow(co) == 70643)
cat('cells per facet:\n'); print(table(co$Batch))

p_a <- ggplot() +
  geom_point(data = co[, c('UMAP_1', 'UMAP_2')], aes(UMAP_1, UMAP_2),
             color = '#E0E0E0', size = 0.12, alpha = 0.35, stroke = 0) +
  geom_point(data = co, aes(UMAP_1, UMAP_2, color = Batch),
             size = 0.15, alpha = 0.8, stroke = 0) +
  scale_color_manual(values = batch_colors, breaks = batch_order, name = NULL) +
  facet_wrap(~Batch, ncol = 4) +
  theme_void(base_size = 7) +
  theme(strip.text = element_text(size = 6.5),
        panel.spacing = unit(0.05, 'cm'), legend.position = 'bottom',
        legend.text = element_text(size = 6),
        legend.key.height = unit(0.16, 'cm'), legend.key.width = unit(0.16, 'cm'))
ggsave(file.path(OUT, 'S2a_batch_shadow_facets.pdf'), p_a,
       width = 16.97, height = 4.97, units = 'cm', device = grDevices::cairo_pdf)
ggsave(file.path(OUT, 'S2a_batch_shadow_facets.png'), p_a,
       width = 16.97, height = 4.97, units = 'cm', dpi = 150)
cat('saved S2a_batch_shadow_facets.pdf\n')
'''
r2 = os.path.join(OUT, 'figS2_s2_panelA.R')
with open(r2, 'w') as fh:
    fh.write(R2)
run_r(r2)

cells per facet:

             10x  HT-scCAT-seq-HT  dsciCAT-seq-50k dsciCAT-seq-100k 
            8438            14762            17093            30350 
saved S2a_batch_shadow_facets.pdf

Rscript exit code 0 - 0.5 min


0

In [7]:
p = os.path.join(OUT, 'S2a_batch_shadow_facets.png')
if os.path.exists(p):
    show(p)

<repo>/figS2/figS2_out/S2a_batch_shadow_facets.png  (152 KB)


**Expected** — one 16.97 × 4.97 cm PDF, four facets of 8,438 / 14,762 / 17,093 /
30,350 coloured points over the same 70,643-point grey shadow. The claim the
panel carries is negative — no library owns a region of the embedding — so what
to look for is coloured points *inside* grey-only territory in every facet.

---

## 3. Panel B — cell-type composition of each technology

**Input**: `technique` + `cell_type_updated` from the §1b cache (metadata only).
**Output**: `S2_OUT/S2b_proportion_by_technique.pdf` (+ `.png`).

One stacked bar per technology, 33 segments in the fixed `dot_order`, numbered
legend identical to panels A/C of Figure 2. Percentages are within a technology
(each bar sums to 100 %).

In [8]:
R3 = r'''
suppressPackageStartupMessages({library(ggplot2); library(dplyr)})
OUT <- Sys.getenv('S2_OUT', 'figS2_out')
S   <- readRDS(file.path(OUT, 's2_light.RDS'))
md  <- S$meta; dot_order <- S$dot_order; num_map <- S$num_map; cols <- S$cols
leg_lab <- setNames(paste0(num_map[dot_order], ' ', dot_order), dot_order)

prop <- md %>% count(technique, cell_type_updated) %>%
  group_by(technique) %>% mutate(proportion = n / sum(n) * 100) %>% ungroup()
tech_levels <- c('10x', 'dsciCAT-seq', 'HT-scCAT-seq-HT')
prop$technique <- factor(prop$technique, levels = tech_levels)

cat('per-technology totals:\n'); print(prop %>% group_by(technique) %>%
  summarise(cells = sum(n), .groups = 'drop'))
cat('\ntypes the legend quotes (%, by technology):\n')
q <- rbind('In_D1', 'AST', 'Oligo', 'COP/NFOL', 'OPC', 'Ex_L2/3_IT', 'Ex_L6_CT')
w <- prop %>% filter(cell_type_updated %in% q) %>%
  select(technique, cell_type_updated, proportion) %>%
  tidyr::pivot_wider(names_from = technique, values_from = proportion)
print(as.data.frame(
  cbind(cell_type_updated = w$cell_type_updated,
        round(w[, tech_levels, drop = FALSE], 1)), row.names = NULL))

p_b <- ggplot(prop, aes(x = technique, y = proportion,
                        fill = factor(cell_type_updated, levels = dot_order))) +
  geom_col(width = 0.68, colour = NA) +
  scale_x_discrete(limits = tech_levels) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.02))) +
  scale_fill_manual(values = cols, breaks = dot_order, labels = leg_lab,
                    name = 'Cell type (n = number)') +
  guides(fill = guide_legend(ncol = 2, keywidth = 0.20, keyheight = 0.20)) +
  labs(x = NULL, y = 'Cell type proportion (%)') +
  theme_bw(base_size = 7) +
  theme(axis.text = element_text(size = 6.5), axis.text.x = element_text(size = 6.5,
        angle = 30, hjust = 1, vjust = 1), panel.grid = element_blank(),
        panel.border = element_rect(linewidth = 0.3, fill = NA),
        legend.position = 'right', legend.text = element_text(size = 6),
        legend.title = element_text(size = 6.5), legend.key = element_blank(),
        legend.background = element_blank(),
        plot.margin = margin(1.5, 1.5, 1.5, 1.5, unit = 'mm'))
ggsave(file.path(OUT, 'S2b_proportion_by_technique.pdf'), p_b,
       width = 8.8, height = 6.55, units = 'cm', device = grDevices::cairo_pdf)
ggsave(file.path(OUT, 'S2b_proportion_by_technique.png'), p_b,
       width = 8.8, height = 6.55, units = 'cm', dpi = 150)
cat('\nsaved S2b_proportion_by_technique.pdf\n')
'''
r3 = os.path.join(OUT, 'figS2_s3_panelB.R')
with open(r3, 'w') as fh:
    fh.write(R3)
run_r(r3)

per-technology totals:
# A tibble: 3 × 2
  technique       cells
  <fct>           <int>
1 10x              8438
2 dsciCAT-seq     47443
3 HT-scCAT-seq-HT 14762

types the legend quotes (%, by technology):
  cell_type_updated  10x dsciCAT-seq HT-scCAT-seq-HT
1               AST  9.2         6.0             1.8
2          COP/NFOL  0.2         0.4             0.4
3        Ex_L2/3_IT  8.6         9.3            12.7
4          Ex_L6_CT  6.9         5.3             9.3
5             In_D1 10.4         0.9             1.6
6               OPC  3.5         3.1             1.5
7             Oligo  6.4        12.3             2.0

saved S2b_proportion_by_technique.pdf

Rscript exit code 0 - 0.1 min


0

In [9]:
p = os.path.join(OUT, 'S2b_proportion_by_technique.png')
if os.path.exists(p):
    show(p)

<repo>/figS2/figS2_out/S2b_proportion_by_technique.png  (33 KB)


**Expected** — three bars of 8,438 / 47,443 / 14,762 cells; the printed table
gives the percentages the legend quotes: 10x In_D1 ≈ 10.4 % and AST ≈ 9.2 %
(dsciCAT-seq ≈ 1.5–2 % and ≈ 4–6 %), dsciCAT-seq Oligo ≈ 12.3 % (≈ 1.7× 10x),
HT Ex_L2/3_IT + Ex_L6_CT ≈ 22.0 % with Ex_L6_CT ≈ 9.3 % the highest of the three,
and COP/NFOL > 0 in all three technologies.

---

## 4. The engine behind panels C and D — 8 throughputs × 20 seeds

**Input**: `3k_dsci/3k_dsci_full_annotCOP.RDS` (the §1a truth object).
**Output**: `S2_OUT/{level}_ARI_NMI.csv` and `S2_OUT/{level}_per_seed_per_type.csv`
for each of the 8 levels.

Each seed is one complete, independent analysis: draw *n* cells without
replacement, keep their full-atlas labels as truth, normalise → 3,000 HVG →
scale → PCA(50) → harmony on `orig.ident` (FY13/FY14) → neighbours on dims 1:50 →
Leiden at resolution 1. Two things come out of each run:

* **global NMI / ARI** — clusters are given a majority-vote label and the
  resulting per-cell partition is scored against the truth partition
  (`aricode`, NMI variant `"max"`; see §0.2);
* **per-type metrics** — from the `cluster × truth` contingency table directly,
  with no relabelling: `n_true` (drawn), `detected` (the type is the plurality
  of some cluster of ≥ 5 cells), `recall`, `precision`, `F1`.

The two embedded scoring functions are the only part of §4 whose correctness is
not visible in the output, so they were checked against an independent
implementation on six random partitions of 200–20,000 cells: **NMI (max variant)
and ARI agree to 1e-10** (and, in the session that produced these CSVs, against
the `aricode` package itself — ARI identical, NMI to ~1e-16). `emb_ari` returns
`NaN` only for the degenerate case where both partitions are all singletons,
which cannot occur here.

The cell is cached **in `S2_OUT` only**: a level whose two CSVs are already there
is skipped unless `S2_FORCE=1`, and `S2_LEVELS` / `S2_SEEDS` shrink the run to a
smoke test. The CSVs sitting in the analysis tree are deliberately *not* treated
as a cache — §4 is the computation this figure claims, and quietly reading back
the table the figure was drawn from would prove nothing. (§5 and §6 do fall back
to those files, so panels C and D can be drawn without paying 3 h again.) Measured cost of
the full 8 × 20: ≈ 3 h at the parallelism below (12 min for 3k, ≈ 20 min per
level, 40k slowest).

In [10]:
R4E = r'''
suppressPackageStartupMessages({library(Seurat); library(harmony); library(parallel)})
DATA <- Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY')))
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
FY   <- file.path(DATA, '3k_dsci')
dir.create(OUT, showWarnings = FALSE, recursive = TRUE)

# ---- scores ---------------------------------------------------------------
# aricode is the reference (NMI default variant = 'max'). It is not always
# installed and cannot be fetched offline, so the embedded implementation below
# is used as the fallback; it was verified against the package on simulated
# partitions (ARI diff = 0, NMI max-variant diff ~ 1e-16).
USE_ARICODE <- requireNamespace('aricode', quietly = TRUE)
cat('scoring backend:', if (USE_ARICODE) 'aricode package' else 'embedded (max variant)', '\n')

emb_nmi <- function(x, y) {
  ax  <- table(x, y); n <- sum(ax)
  pxy <- ax / n
  px  <- rowSums(ax) / n; py <- colSums(ax) / n
  ent <- function(p) -sum(p[p > 0] * log2(p[p > 0]))
  mi  <- sum(pxy[pxy > 0] * log2(pxy[pxy > 0] /
            as.vector(outer(px, py, '*'))[pxy > 0]))
  mi / max(ent(px), ent(py))
}
emb_ari <- function(x, y) {
  tab <- table(x, y); n <- sum(tab)
  a <- rowSums(tab); b <- colSums(tab)
  C2 <- function(v) sum(v * (v - 1) / 2)
  obs <- C2(as.vector(tab)); expd <- C2(a) * C2(b) / C2(n)
  maxd <- 0.5 * (C2(a) + C2(b))
  (obs - expd) / (maxd - expd)
}
score <- function(a, b) {
  if (!USE_ARICODE) return(list(NMI = emb_nmi(a, b), ARI = emb_ari(a, b)))
  lev <- union(as.character(unique(a)), as.character(unique(b)))   # aricode wants
  ia  <- match(as.character(a), lev); ib <- match(as.character(b), lev)  # integer codes
  return(list(NMI = aricode::NMI(ia, ib), ARI = aricode::ARI(ia, ib)))
}

# ---- per-type metrics from the contingency table --------------------------
per_type_metrics <- function(truth, clust, min_cluster_size = 5) {
  ct <- table(clust, truth)
  if (ncol(ct) == 0) return(NULL)
  plurality <- colnames(ct)[apply(ct, 1, which.max)]
  names(plurality) <- rownames(ct)
  pred <- plurality[as.character(clust)]
  types <- sort(unique(truth))
  do.call(rbind, lapply(types, function(t) {
    tp <- sum(pred == t & truth == t); fp <- sum(pred == t & truth != t)
    fn <- sum(pred != t & truth == t)
    det <- any(apply(ct, 1, function(r) colnames(ct)[which.max(r)] == t) &
                 rowSums(ct) >= min_cluster_size)
    rec  <- ifelse(tp + fn > 0, tp / (tp + fn), NA)
    prec <- ifelse(tp + fp > 0, tp / (tp + fp), NA)
    f1   <- ifelse(is.na(rec) | is.na(prec) | rec + prec == 0, NA,
                   2 * rec * prec / (rec + prec))
    data.frame(type = t, n_true = tp + fn, detected = as.integer(det),
               recall = rec, precision = prec, F1 = f1, stringsAsFactors = FALSE)
  }))
}

# ---- one down-sample = one full re-analysis -------------------------------
Process_RNA <- function(proj, nfeatures = 3000, pc.nums = 50) {
  DefaultAssay(proj) <- 'RNA'
  proj <- NormalizeData(proj)
  proj <- FindVariableFeatures(proj, nfeatures = nfeatures)
  proj <- ScaleData(proj, verbose = FALSE)
  proj <- RunPCA(proj, npcs = pc.nums, reduction.name = 'pca', verbose = FALSE)
  proj
}
run_one <- function(seu, df_full, target_cell_count, seed) {
  set.seed(seed)
  cells <- sample(colnames(seu), size = target_cell_count, replace = FALSE)
  obj <- Process_RNA(subset(seu, cells = cells))
  obj <- RunHarmony(obj, group.by.vars = 'orig.ident', assay.use = 'RNA',
                    reduction.use = 'pca', plot_convergence = FALSE)
  obj <- FindNeighbors(obj, reduction = 'harmony', dims = 1:50, verbose = FALSE)
  obj <- FindClusters(obj, algorithm = 3, resolution = 1, verbose = FALSE)
  clust <- as.character(obj$RNA_snn_res.1)
  truth <- as.character(obj$cell_type)

  ct <- table(clust, truth)
  auto <- colnames(ct)[apply(ct, 1, which.max)]; names(auto) <- rownames(ct)
  sub <- data.frame(cellname = colnames(obj), Cell_type = as.character(auto[clust]))
  both <- merge(df_full, sub, by = 'cellname', all = FALSE)
  stopifnot(nrow(both) == length(clust))
  s <- score(both$Cell_type.x, both$Cell_type.y)
  pt <- per_type_metrics(truth, clust)
  list(seed = seed, NMI = s$NMI, ARI = s$ARI, pt = pt)
}

# ---- the 8 x 20 loop, cached ---------------------------------------------
LEVELS  <- c('3k','5k','10k','15k','20k','25k','30k','40k')
# parallelism per level, as used by the run that produced the delivered CSVs:
# memory-bound, not core-bound (each worker holds its own copy of the object)
NCORES  <- c('3k'=20,'5k'=16,'10k'=12,'15k'=10,'20k'=8,'25k'=6,'30k'=5,'40k'=4)
if (nzchar(Sys.getenv('S2_NCORES'))) NCORES[] <- as.integer(Sys.getenv('S2_NCORES'))
if (nzchar(Sys.getenv('S2_LEVELS')))
  LEVELS <- intersect(LEVELS, strsplit(Sys.getenv('S2_LEVELS'), ',')[[1]])
seeds  <- if (nzchar(Sys.getenv('S2_SEEDS')))
            as.integer(strsplit(Sys.getenv('S2_SEEDS'), ',')[[1]]) else seq(1, 200, by = 10)
FORCE  <- Sys.getenv('S2_FORCE') == '1'

todo <- LEVELS[sapply(LEVELS, function(lv) FORCE || !(
  file.exists(file.path(OUT, paste0(lv, '_ARI_NMI.csv'))) &&
  file.exists(file.path(OUT, paste0(lv, '_per_seed_per_type.csv')))))]
cat('levels to run:', if (length(todo)) paste(todo, collapse = ',') else 'none (cache complete)',
    '| seeds:', length(seeds), '\n')

if (length(todo)) {
  seu <- readRDS(file.path(FY, '3k_dsci_full_annotCOP.RDS'))
  for (a in setdiff(Assays(seu), 'RNA')) seu[[a]] <- NULL     # RAM
  gc()
  df_full <- data.frame(cellname = rownames(seu@meta.data),
                        Cell_type = as.character(seu@meta.data$cell_type))
  cat('truth object:', ncol(seu), 'cells |', length(unique(df_full$Cell_type)),
      'types | COP/NFOL:', sum(df_full$Cell_type == 'COP/NFOL'), '\n')

  for (lv in todo) {
    n <- as.integer(gsub('k', '000', lv)); t0 <- Sys.time()
    res <- mclapply(seeds, function(s) tryCatch(run_one(seu, df_full, n, s),
                       error = function(e) { cat('[ERROR]', lv, s, conditionMessage(e), '\n'); NULL }),
                    mc.cores = min(NCORES[[lv]], length(seeds)), mc.preschedule = FALSE)
    # mclapply silently drops a worker that dies (OOM / fork instability) --
    # those seeds come back as NULL from the tryCatch OR as missing elements.
    # Retry any seed that did not deliver, one at a time, in the parent process.
    retry <- seeds[!vapply(seq_along(seeds), function(k)
              !is.null(res[[k]]) && !is.null(res[[k]]$seed), logical(1))]
    if (length(retry)) {
      cat('[retry]', length(retry), 'seeds:', paste(retry, collapse = ','), '\n')
      for (s in retry) {
        res[[match(s, seeds)]] <- tryCatch(run_one(seu, df_full, n, s),
                     error = function(e) { cat('[ERROR]', lv, s, conditionMessage(e), '\n'); NULL })
      }
    }
    ok  <- Filter(Negate(is.null), res)
    met <- do.call(rbind, lapply(ok, function(o) data.frame(Seed = o$seed, NMI = o$NMI, ARI = o$ARI)))
    pt  <- do.call(rbind, lapply(ok, function(o) { p <- o$pt; p$Seed <- o$seed; p$Level <- lv; p }))
    write.csv(met, file.path(OUT, paste0(lv, '_ARI_NMI.csv')), row.names = FALSE)
    write.csv(pt,  file.path(OUT, paste0(lv, '_per_seed_per_type.csv')), row.names = FALSE)
    cat(sprintf('[%s] %d/%d seeds in %.1f min | NMI %.3f+-%.3f | ARI %.3f+-%.3f | rows %d\n',
                lv, length(ok), length(seeds),
                as.numeric(difftime(Sys.time(), t0, units = 'mins')),
                mean(met$NMI), sd(met$NMI), mean(met$ARI), sd(met$ARI), nrow(pt)))
  }
}
''';
r4e = os.path.join(OUT, 'figS2_s4_engine.R')
with open(r4e, 'w') as fh:
    fh.write(R4E)
rc = run_r(r4e)
assert rc == 0, '§4 failed'

scoring backend: aricode package 
levels to run: none (cache complete) | seeds: 20 

Rscript exit code 0 - 0.1 min


**Expected** — per level, 20 rows in `{level}_ARI_NMI.csv` and ≤ 740 rows in
`{level}_per_seed_per_type.csv` (37 types × 20 seeds minus the types never
drawn; the delivered 3k file has 739). The means ± SD these CSVs give, which are
the values of the delivered panels:

| level | NMI | ARI | mean recall | | level | NMI | ARI | mean recall |
|---|---|---|---|---|---|---|---|---|
| 3k | 0.726 ± 0.019 | 0.652 ± 0.028 | 0.447 | | 20k | 0.884 ± 0.011 | 0.853 ± 0.022 | 0.789 |
| 5k | 0.794 ± 0.016 | 0.731 ± 0.023 | 0.577 | | 25k | 0.894 ± 0.009 | 0.865 ± 0.017 | 0.814 |
| 10k | 0.850 ± 0.010 | 0.806 ± 0.018 | 0.707 | | 30k | 0.906 ± 0.010 | 0.887 ± 0.015 | 0.839 |
| 15k | 0.873 ± 0.010 | 0.838 ± 0.019 | 0.768 | | 40k | 0.922 ± 0.010 | 0.906 ± 0.019 | 0.872 |

Both scores rise monotonically and their SD shrinks; the marginal gain is
already small by 10–15 k, which is panel C's point. A single seed is *not*
monotone for a rare type (§0.2), so do not read one run of one level as a
result.

---

## 5. Panel C — clustering consistency against throughput

**Input**: the 8 `{level}_ARI_NMI.csv` of §4.
**Output**: `S2_OUT/S2c_NMI_ARI_vs_throughput.pdf` (+ `.png`).

Mean ± SD over the 20 seeds, x on a log scale at the 8 analysed cell numbers.

In [11]:
R5 = r'''
suppressPackageStartupMessages({library(ggplot2); library(dplyr); library(tidyr)})
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
FY   <- file.path(Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY'))), '3k_dsci')
LEVELS <- c('3k','5k','10k','15k','20k','25k','30k','40k')
LEVEL_N <- as.integer(gsub('k', '000', LEVELS))
read_lv <- function(lv) {
  f <- file.path(OUT, paste0(lv, '_ARI_NMI.csv'))
  if (!file.exists(f)) f <- file.path(FY, paste0('3k_dsci_down', lv, '_ARI_NMI.csv'))
  read.csv(f)
}
dN <- do.call(rbind, lapply(seq_along(LEVELS), function(i) {
  x <- read_lv(LEVELS[i])
  data.frame(Level = LEVELS[i], n = LEVEL_N[i], NMI = x$NMI, ARI = x$ARI) }))
cat('seeds per level:', paste(table(dN$Level), collapse = ' '), '\n')
s <- dN %>% group_by(Level, n) %>%
  summarise(nmi_m = mean(NMI), nmi_sd = sd(NMI), ari_m = mean(ARI), ari_sd = sd(ARI),
            .groups = 'drop') %>%
  pivot_longer(c(nmi_m, ari_m), names_to = 'stat', values_to = 'm')
s$sd    <- ifelse(s$stat == 'nmi_m', s$nmi_sd, s$ari_sd)
s$label <- ifelse(s$stat == 'nmi_m', 'NMI', 'ARI')
print(s %>% group_by(Level) %>% summarise(NMI = mean(m[label == 'NMI']),
                                          ARI = mean(m[label == 'ARI']), .groups = 'drop'))

p_c <- ggplot(s, aes(x = n, y = m, color = label)) +
  geom_errorbar(aes(ymin = m - sd, ymax = m + sd), linewidth = 0.25, width = 0.03) +
  geom_line(linewidth = 0.5) + geom_point(size = 1.1) +
  scale_x_log10(breaks = LEVEL_N, labels = LEVELS) +
  scale_color_manual(values = c(NMI = '#E41A1C', ARI = '#377EB8')) +
  labs(x = 'Downsample throughput (cells)', y = 'Score') +
  guides(color = guide_legend(title = NULL, keywidth = 0.25, keyheight = 0.25)) +
  theme_bw(base_size = 7) +
  theme(axis.text = element_text(size = 6.5), axis.text.x = element_text(size = 6.5,
        angle = 45, hjust = 1), axis.title = element_text(size = 7),
        panel.grid = element_blank(), panel.border = element_rect(linewidth = 0.3, fill = NA),
        legend.position = 'right', legend.text = element_text(size = 6.5),
        legend.key.size = unit(0.28, 'cm'),
        plot.margin = margin(1.5, 1.5, 1.5, 1.5, unit = 'mm'))
ggsave(file.path(OUT, 'S2c_NMI_ARI_vs_throughput.pdf'), p_c,
       width = 8.8, height = 6.55, units = 'cm', device = grDevices::cairo_pdf)
ggsave(file.path(OUT, 'S2c_NMI_ARI_vs_throughput.png'), p_c,
       width = 8.8, height = 6.55, units = 'cm', dpi = 150)
cat('saved S2c_NMI_ARI_vs_throughput.pdf\n')
'''
r5 = os.path.join(OUT, 'figS2_s5_panelC.R')
with open(r5, 'w') as fh:
    fh.write(R5)
run_r(r5)

seeds per level: 20 20 20 20 20 20 20 20 
# A tibble: 8 × 3
  Level   NMI   ARI
  <chr> <dbl> <dbl>
1 10k   0.850 0.806
2 15k   0.873 0.838
3 20k   0.884 0.853
4 25k   0.894 0.865
5 30k   0.906 0.887
6 3k    0.726 0.652
7 40k   0.922 0.906
8 5k    0.794 0.731
saved S2c_NMI_ARI_vs_throughput.pdf

Rscript exit code 0 - 0.0 min


0

In [12]:
p = os.path.join(OUT, 'S2c_NMI_ARI_vs_throughput.png')
if os.path.exists(p):
    show(p)

<repo>/figS2/figS2_out/S2c_NMI_ARI_vs_throughput.png  (25 KB)


---

## 6. Panel D — which cell types depth actually buys

**Input**: the 8 `{level}_per_seed_per_type.csv` of §4 + the §1a composition.
**Output**: `S2_OUT/S2d_pertype_recall_heatmap.pdf` (+ `.png`) and
`S2_OUT/FigD_paired_recall_3k_vs_40k.csv`.

Each tile is the mean recall of one type at one throughput over the 20 seeds.
The x order is the *full-data abundance* order (rarest type first), so the
left-hand side of the panel is where depth has to work. The types whose labels
are marked are those that improve with depth by a **paired** test: the same seed
at 3k and at 40k, Wilcoxon signed-rank over the 20 pairs, BH-FDR < 0.05 and
Δrecall ≥ 0.5. The pairing is only possible because every level uses the same
seed list.

In [13]:
R6 = r'''
suppressPackageStartupMessages({library(ggplot2); library(dplyr); library(tidyr)
  library(stringr)})
OUT  <- Sys.getenv('S2_OUT', 'figS2_out')
FY   <- file.path(Sys.getenv('S2_DATA', os.environ.get('DSCI_FY_ROOT', os.path.join(os.environ.get('DSCI_WORK_ROOT', ''), 'FY'))), '3k_dsci')
LEVELS <- c('3k','5k','10k','15k','20k','25k','30k','40k')
read_lv <- function(lv, what) {
  f <- file.path(OUT, paste0(lv, what))
  if (!file.exists(f)) f <- file.path(FY, paste0(if (grepl('ARI', what)) '3k_dsci_down' else '', lv, what))
  read.csv(f)
}
d <- do.call(rbind, lapply(LEVELS, function(lv) {
  x <- read_lv(lv, '_per_seed_per_type.csv'); x$Level <- lv; x }))
cat('rows:', nrow(d), '| types:', length(unique(d$type)), '| levels:',
    length(unique(d$Level)), '\n')
cat('rows per level:', paste(sapply(LEVELS, function(lv) sum(d$Level == lv)),
                             collapse = ' '), '\n')

# the composition §1a wrote, else the delivered copy of it
fc <- file.path(OUT, 'truth_composition.csv')
if (!file.exists(fc)) fc <- file.path(FY, '3k_dsci_full_annotCOP_meancelltype.csv')
comp <- read.csv(fc)
comp$ab <- comp$Freq / sum(comp$Freq)
d <- merge(d, comp[, c('type', 'ab')], by = 'type', all.x = TRUE)

# ---- paired test, recomputed here (not read from a table) -----------------
a <- d %>% filter(Level == '3k')  %>% select(Seed, type, recall)
b <- d %>% filter(Level == '40k') %>% select(Seed, type, recall)
m <- merge(a, b, by = c('Seed', 'type'), suffixes = c('_3k', '_40k'))
res <- m %>% group_by(type) %>%
  summarise(diff = mean(recall_40k - recall_3k),
            pairs = n(),
            p = tryCatch(suppressWarnings(
                   wilcox.test(recall_3k, recall_40k, paired = TRUE)$p.value),
                   error = function(e) NA_real_), .groups = 'drop') %>%
  mutate(padj = p.adjust(p, method = 'BH'))
write.csv(res, file.path(OUT, 'FigD_paired_recall_3k_vs_40k.csv'), row.names = FALSE)
sig <- res$type[which(res$padj < 0.05 & res$diff >= 0.5)]   # which(): NA-safe
cat('marked types (BH-FDR < 0.05 & delta >= 0.5):', length(sig), '\n')
cat(sort(sig), collapse = ', ', '\n')
cat('mean recall by level:', paste(sprintf('%s %.3f', LEVELS,
    sapply(LEVELS, function(lv) mean(d$recall[d$Level == lv], na.rm = TRUE))),
    collapse = '  '), '\n')
cat('COP/NFOL recall by level:', paste(sprintf('%s %.2f', LEVELS,
    sapply(LEVELS, function(lv) mean(d$recall[d$Level == lv & d$type == 'COP/NFOL'],
                                      na.rm = TRUE))), collapse = '  '), '\n')

# ---- the heatmap ----------------------------------------------------------
dd <- d %>% group_by(type, Level) %>%
  summarise(m = mean(recall, na.rm = TRUE), ab = first(ab), .groups = 'drop')
ord <- dd %>% arrange(ab) %>% pull(type) %>% unique()
dd$type  <- factor(dd$type, levels = ord)
dd$Level <- factor(dd$Level, levels = LEVELS)
wrapped <- str_wrap(ord, width = 10)
have_ggtext <- requireNamespace('ggtext', quietly = TRUE)
if (have_ggtext) {
  lab <- ifelse(ord %in% sig, sprintf("<span style='color:#B2182B'>%s</span>", wrapped),
                sprintf("<span style='color:grey25'>%s</span>", wrapped))
} else {
  cat('[note] ggtext absent -> significant labels get a "+" marker instead of red ink\n')
  lab <- ifelse(ord %in% sig, paste0(wrapped, ' +'), wrapped)
}
p_d <- ggplot(dd, aes(x = type, y = Level, fill = m)) +
  geom_tile(color = 'white', linewidth = 0.2) +
  scale_fill_gradient2(low = '#2166AC', mid = '#F7F7F7', high = '#B2182B',
                       midpoint = 0.5, na.value = 'grey92', name = 'Mean\nrecall') +
  scale_x_discrete(labels = setNames(lab, ord)) +
  labs(x = NULL, y = NULL) +
  theme_bw(base_size = 7) +
  theme(axis.text.x = if (have_ggtext) ggtext::element_markdown(angle = 90, hjust = 0.5,
            vjust = 0.5, size = 6, lineheight = 0.85) else
            element_text(angle = 90, hjust = 0.5, vjust = 0.5, size = 6,
                         lineheight = 0.85, color = 'grey25'),
        axis.text.y = element_text(size = 6.5), axis.ticks = element_blank(),
        panel.border = element_blank(), panel.grid = element_blank(),
        legend.position = 'right', legend.title = element_text(size = 6.5),
        legend.text = element_text(size = 6), legend.key.size = unit(0.28, 'cm'),
        plot.margin = margin(1.5, 1.5, 1.5, 1.5, unit = 'mm'))
ggsave(file.path(OUT, 'S2d_pertype_recall_heatmap.pdf'), p_d,
       width = 16.97, height = 4.90, units = 'cm', device = grDevices::cairo_pdf)
ggsave(file.path(OUT, 'S2d_pertype_recall_heatmap.png'), p_d,
       width = 16.97, height = 4.90, units = 'cm', dpi = 150)
cat('saved S2d_pertype_recall_heatmap.pdf\n')
'''
r6 = os.path.join(OUT, 'figS2_s6_panelD.R')
with open(r6, 'w') as fh:
    fh.write(R6)
run_r(r6)

rows: 5919 | types: 37 | levels: 8 
rows per level: 739 740 740 740 740 740 740 740 
marked types (BH-FDR < 0.05 & delta >= 0.5): 16 
EC Ependyma Ex_CAR3 Ex_CR Ex_HY-MM Ex_IC Ex_L2/3_IT_PIR Ex_L4 Ex_L5_NP Ex_L6_IT Ex_L6b In_HAP1 In_MEIS2 In_PVALB In_SST In_TH ,  
mean recall by level: 3k 0.447  5k 0.577  10k 0.707  15k 0.768  20k 0.789  25k 0.814  30k 0.839  40k 0.872 
COP/NFOL recall by level: 3k 0.00  5k 0.17  10k 0.38  15k 0.41  20k 0.42  25k 0.47  30k 0.47  40k 0.50 
saved S2d_pertype_recall_heatmap.pdf

Rscript exit code 0 - 0.1 min


0

In [14]:
p = os.path.join(OUT, 'S2d_pertype_recall_heatmap.png')
if os.path.exists(p):
    show(p)

<repo>/figS2/figS2_out/S2d_pertype_recall_heatmap.png  (36 KB)


**Expected** — 37 columns × 8 rows; mean recall 0.447 (3k) → 0.872 (40k);
**16** marked types: EC, Ependyma, Ex_CAR3, Ex_CR, Ex_HY-MM, Ex_IC,
Ex_L2/3_IT_PIR, Ex_L4, Ex_L5_NP, Ex_L6_IT, Ex_L6b, In_HAP1, In_MEIS2, In_PVALB,
In_SST, In_TH. **COP/NFOL is not among the 16** and the reason is worth knowing
before anyone "fixes" it: its paired Δrecall is **0.4965** (recall 0.00 at 3k in
all 20 seeds → 0.497 at 40k, BH-FDR 1.5e-4), so it clears significance and misses
the Δ ≥ 0.5 cut-off by 0.0035. The threshold, not the biology, is what left the
figure's rarest type unlabelled — quote the Δ if this panel is discussed.
The reading of the panel: overall consistency saturates early (panel C) while
rare-type recovery keeps paying up to 30–40k (panel D) — the two panels are
complementary, not redundant.

---

## Requirements

* R ≥ 4.1 with `Seurat` (≥ 4), `harmony`, `ggplot2` (≥ 3.4, `linewidth`),
  `dplyr`, `tidyr`, `stringr`, `pals`, `parallel`.
  `aricode` for §4's scores (without it the embedded max-variant implementation
  runs, verified identical to the package — see §0.2); `ggtext` only changes how
  panel D marks its 16 significant labels (red ink vs a `+` suffix).
* Python ≥ 3.8 with `numpy` + `pandas` (PNG preview conversion only) and
  nothing is displayed inline; figures are saved and reported by `show()` (keep the notebook small on GitHub).
* Disk: the two input objects are 13.2 + 5.4 GB and are read, not copied.
  §4's cache is ~1 MB.
* RAM: §1b ≈ 14 GB for the integrated object; §4's full run was measured at
  20 workers on 3k down to 4 workers on 40k within ~100 GB — lower
  `S2_NCORES`/the per-level table if the host is smaller.
* Time: §1b 5 min (measured), §2–§3 a couple of minutes, §5–§6 under a minute
  from the cache.
  **§4 from scratch ≈ 3 h on the host that produced the delivered CSVs**
  (12 min at 3k); measured 2026-10-23 on conda R 4.2.2: ≈ 52 min at 3k.
  Keep `--ExecutePreprocessor.timeout=-1` so the kernel does not kill it.

## Run

```bash
export S2_DATA=/path/to/FY S2_OUT=figS2_out

# cheap smoke run first: one level, two seeds, no truth rebuild
S2_LEVELS=3k S2_SEEDS=1,11 jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=-1 Figure_S2.ipynb

# full reproduction (reuses §4's CSV cache; S2_FORCE=1 recomputes it)
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=-1 Figure_S2.ipynb
```

A smoke run checks the plumbing only: panels C and D drawn from 2 seeds are not
the figure — the bands of §5 and the whole marking of §6 need all 20 seeds.

## Status of this notebook

Executed end-to-end 2026-10-23 in a networked conda R 4.2.2 environment
(Seurat 4.3.0, harmony 1.2.3, `aricode` 1.0.1 as the §4 scoring backend,
`ggtext` present): **0 errors**. §4 ran at the 3k level (20/20 seeds; the
engine retries mclapply-dropped workers single-process), the other levels
reused the delivered cache, and §5/§6 matched every delivered number to
machine precision. §1a derives the 269-cell COP/NFOL split from the
pre-split `3k_dsci_full.RDS` and cross-checks labels against the post-split
marker export. `README.md` holds the accounting.